# RAG Failure-Mode Experiment — Python Stdlib Docs
### Fully local / free-tier Colab (T4 GPU), no paid APIs

**Pipeline:** build corpus → build FAISS index → baseline retrieval+generation → baseline judging
→ save → apply reranker intervention → re-run+judge → save → before/after comparison → analysis.

**Models**
- Generator: `Qwen/Qwen2.5-3B-Instruct` (4-bit, bitsandbytes) — originally spec'd as
  `meta-llama/Llama-3.2-3B-Instruct`, swapped because that repo is gated on Hugging Face and
  requires manual license acceptance + an auth token; Qwen2.5-3B-Instruct is ungated and
  comparable in capability for this task.
- Judge: `Qwen/Qwen2.5-7B-Instruct` (4-bit, bitsandbytes) — **a different, larger-scale model
  than the generator**, because a model judging its own outputs tends to under-detect its own
  reasoning errors (subtle misuse/hallucination especially). Note both generator and judge are
  now Qwen-family (3B vs. 7B) rather than cross-family — this is a minor limitation versus the
  original cross-family design, discussed further in Section 10.
- Embedder: `sentence-transformers/all-MiniLM-L6-v2` (CPU) — swap to `BAAI/bge-small-en-v1.5` by
  changing one constant below if you prefer.
- Reranker (intervention): `cross-encoder/ms-marco-MiniLM-L-6-v2` (CPU)

**Design note on retrieval-miss:** rather than relying on the judge to decide whether retrieval
"missed", we compute this **objectively**: each eval question carries a `gold_section_id` that
must match a real chunk id in the corpus. If that id isn't in the top-k retrieved chunks shown to
the generator, we label the case `retrieval-miss` programmatically — no LLM judgment needed for
that call. The judge is only asked to distinguish between the remaining four categories
(`correct` / `retrieved-but-ignored` / `retrieved-but-misused` / `hallucinated`) for the cases
where the gold chunk *was* retrieved. This is more rigorous than an all-LLM-judged pipeline and
plays to your point that these are verifiable documentation facts.

**Resumability:** every result (corpus, index, generations, judgments) is written to Google Drive
after each step/item, and the eval loops skip ids that already have a saved result — safe to
re-run after a disconnect.


## 0. Setup: installs, Drive mount, paths

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate bitsandbytes beautifulsoup4 requests matplotlib pandas
print("done")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/rag_failure_modes'
os.makedirs(PROJECT_DIR, exist_ok=True)

CORPUS_PATH        = f'{PROJECT_DIR}/corpus.json'
INDEX_PATH         = f'{PROJECT_DIR}/faiss.index'
INDEX_META_PATH    = f'{PROJECT_DIR}/faiss_meta.json'
EVAL_SET_PATH      = f'{PROJECT_DIR}/eval_questions.json'   # <-- you upload/provide this (see section 2)
BASELINE_GEN_PATH  = f'{PROJECT_DIR}/baseline_generations.json'
BASELINE_JUDGE_PATH= f'{PROJECT_DIR}/baseline_judgments.json'
INTERV_GEN_PATH    = f'{PROJECT_DIR}/intervention_generations.json'
INTERV_JUDGE_PATH  = f'{PROJECT_DIR}/intervention_judgments.json'
SPOTCHECK_PATH     = f'{PROJECT_DIR}/spotcheck_sample.csv'

print('Project dir:', PROJECT_DIR)

In [ ]:
import json, gc, time, re, random
import numpy as np
import torch

def load_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default if default is not None else {}

def save_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize() if torch.cuda.is_available() else None


## 1. Build the corpus: Python stdlib docs

Scrapes the official docs pages for a fixed list of modules and splits each page into chunks at
the `dl`/definition level (functions, classes, exceptions) plus intro paragraphs, so each chunk
is a citable, addressable unit with a stable id like `os.path#os.path.join`.

Re-running this cell is safe — it always rebuilds `corpus.json` from scratch (scraping is cheap
and we want a reproducible, versioned corpus saved to Drive).


In [ ]:
import requests
from bs4 import BeautifulSoup

MODULES = ['os', 'os.path', 'itertools', 'collections', 're', 'json', 'functools']
BASE_URL = 'https://docs.python.org/3/library/{}.html'

def scrape_module(mod):
    url = BASE_URL.format(mod)
    resp = requests.get(url, timeout=20, headers={'User-Agent': 'rag-experiment/1.0'})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    body = soup.find('div', {'class': 'body'}) or soup.find('article') or soup
    chunks = []

    # Each <dl class="py ..."> is one definition (function/class/exception/data)
    for dl in body.find_all('dl'):
        dt = dl.find('dt')
        if dt is None:
            continue
        anchor = dt.get('id') or (dt.find('a', class_='headerlink') and dt.find_previous(id=True))
        anchor_id = dt.get('id')
        if not anchor_id:
            continue
        # signature text
        sig = dt.get_text(' ', strip=True)
        sig = re.sub(r'\s+', ' ', sig)
        dd = dt.find_next_sibling('dd')
        body_text = dd.get_text(' ', strip=True) if dd else ''
        body_text = re.sub(r'\s+', ' ', body_text)
        full_text = f"{sig}\n{body_text}".strip()
        if len(full_text) < 20:
            continue
        chunks.append({
            'id': f'{mod}#{anchor_id}',
            'module': mod,
            'title': sig[:200],
            'url': f'{url}#{anchor_id}',
            'text': full_text[:3000],
        })

    # Also grab top-level intro paragraphs (module-level description, not tied to one function)
    intro_paras = []
    for p in body.find_all('p', limit=15):
        if p.find_parent('dl'):
            continue
        txt = p.get_text(' ', strip=True)
        if len(txt) > 60:
            intro_paras.append(txt)
        if len(intro_paras) >= 3:
            break
    if intro_paras:
        chunks.append({
            'id': f'{mod}#intro',
            'module': mod,
            'title': f'{mod} module overview',
            'url': url,
            'text': ' '.join(intro_paras)[:3000],
        })

    return chunks

corpus = []
for m in MODULES:
    try:
        mod_chunks = scrape_module(m)
        print(f'{m}: {len(mod_chunks)} chunks')
        corpus.extend(mod_chunks)
        time.sleep(0.5)  # be polite to docs.python.org
    except Exception as e:
        print(f'FAILED {m}: {e}')

print(f'\nTotal chunks: {len(corpus)}')
save_json(corpus, CORPUS_PATH)
print('Saved corpus to', CORPUS_PATH)

In [ ]:
# Sanity check: how many chunks per module, and a peek at one
from collections import Counter
corpus = load_json(CORPUS_PATH, [])
print(Counter(c['module'] for c in corpus))
print()
print(corpus[0]['id'], '\n', corpus[0]['text'][:300])

## 2. Load the eval set

**This notebook does not build the eval set interactively.** Provide a JSON file with 50-80
question objects, upload it to `EVAL_SET_PATH` on Drive (or to Colab's local filesystem and copy
it there), and this cell loads it. Constructing these semi-manually against the docs (so gold
answers are directly verifiable, not just LLM-judged) is the recommended workflow — see the
`eval_questions_template.json` file provided alongside this notebook for the exact schema and a
handful of worked examples to extend from.

Required fields per item:
```json
{
  "id": "q001",
  "question": "What does the 'default' argument of functools.reduce return if the iterable is empty and no default is given?",
  "gold_answer": "TypeError is raised.",
  "gold_section_id": "functools#functools.reduce",
  "gold_module": "functools"
}
```
`gold_section_id` **must** match a chunk `id` produced in Section 1 exactly — this is what makes
retrieval-miss detection objective rather than LLM-judged. Run the cell below after uploading;
it will tell you if any `gold_section_id` doesn't match a real corpus chunk so you can fix typos
before running the (expensive) generation loop.


In [ ]:
eval_set = load_json(EVAL_SET_PATH, None)
if eval_set is None:
    raise FileNotFoundError(
        f'No eval set found at {EVAL_SET_PATH}. Upload your eval_questions.json to that Drive path '
        f'(see eval_questions_template.json for the schema) and re-run this cell.'
    )

corpus_ids = {c['id'] for c in corpus}
bad = [q['id'] for q in eval_set if q['gold_section_id'] not in corpus_ids]
print(f'Loaded {len(eval_set)} eval questions.')
if bad:
    print(f'WARNING: {len(bad)} questions reference a gold_section_id not present in the corpus:')
    print(bad)
    print('Fix these (typo in id, or module/section not scraped) before proceeding.')
else:
    print('All gold_section_id values matched corpus chunks. Good to go.')

## 3. Embed corpus + build FAISS index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # swap to 'BAAI/bge-small-en-v1.5' if preferred

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device='cpu')

def build_index(corpus):
    texts = [c['text'] for c in corpus]
    emb = embed_model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    emb = np.asarray(emb, dtype='float32')
    index = faiss.IndexFlatIP(emb.shape[1])  # inner product on normalized vectors = cosine similarity
    index.add(emb)
    return index

if os.path.exists(INDEX_PATH) and os.path.exists(INDEX_META_PATH):
    index = faiss.read_index(INDEX_PATH)
    meta = load_json(INDEX_META_PATH)
    print(f'Loaded existing index with {index.ntotal} vectors from Drive.')
else:
    index = build_index(corpus)
    faiss.write_index(index, INDEX_PATH)
    meta = {'chunk_ids': [c['id'] for c in corpus], 'embed_model': EMBED_MODEL_NAME}
    save_json(meta, INDEX_META_PATH)
    print(f'Built and saved new index with {index.ntotal} vectors.')

chunk_by_id = {c['id']: c for c in corpus}
id_order = meta['chunk_ids']  # position i in the index == id_order[i]

In [ ]:
def dense_retrieve(query, k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        cid = id_order[idx]
        results.append({'id': cid, 'score': float(score), 'chunk': chunk_by_id[cid]})
    return results

# quick smoke test
for r in dense_retrieve('what does itertools.chain do', k=3):
    print(round(r['score'], 3), r['id'])

## 4. Model loading helpers (one model on GPU at a time)

`load_generator()` / `load_judge()` load their respective model in 4-bit; `unload_current()`
deletes whatever is currently loaded and clears CUDA cache before the next load. The eval loops
below always: load generator → generate ALL answers for a phase → unload → load judge → judge ALL
answers for that phase → unload. This keeps peak VRAM to a single small model at a time.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_MODEL_NAME   = 'Qwen/Qwen2.5-3B-Instruct'  # swapped from Llama-3.2-3B-Instruct (gated repo, needs HF auth)
JUDGE_MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

_current = {'model': None, 'tok': None, 'name': None}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def unload_current():
    if _current['model'] is not None:
        del _current['model']
        del _current['tok']
        _current['model'] = None
        _current['tok'] = None
        _current['name'] = None
    free_gpu()
    print('Unloaded. Free GPU mem (GB):', torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 'n/a')

def _load(name):
    if _current['name'] == name:
        return _current['model'], _current['tok']
    unload_current()
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb_config, device_map='auto')
    model.eval()
    _current.update(model=model, tok=tok, name=name)
    print(f'Loaded {name}. Free GPU mem (GB):', torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 'n/a')
    return model, tok

def load_generator():
    return _load(GEN_MODEL_NAME)

def load_judge():
    return _load(JUDGE_MODEL_NAME)

@torch.no_grad()
def chat_generate(model, tok, system, user, max_new_tokens=300, temperature=0.0):
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=max(temperature, 1e-5),
        pad_token_id=tok.eos_token_id,
    )
    text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

## 5. Generation loop (baseline retrieval: plain dense top-3)

Resumable: results are saved to `BASELINE_GEN_PATH` after every question, and already-completed
ids are skipped on re-run.


In [ ]:
GEN_SYSTEM_PROMPT = (
    "You are a documentation assistant. Answer the user's question ONLY using the provided "
    "CONTEXT excerpts from the Python standard library documentation. If the context does not "
    "contain the answer, say you don't know rather than guessing. Be concise and precise, and cite "
    "which module the answer came from if relevant."
)

def format_context(retrieved):
    blocks = []
    for r in retrieved:
        blocks.append(f"[{r['id']}]\n{r['chunk']['text']}")
    return '\n\n'.join(blocks)

def run_generation_phase(eval_set, retrieve_fn, top_k, save_path, phase_label):
    results = load_json(save_path, {})
    model, tok = load_generator()
    for q in eval_set:
        if q['id'] in results:
            continue
        retrieved = retrieve_fn(q['question'], k=top_k)
        retrieved_ids = [r['id'] for r in retrieved]
        context = format_context(retrieved)
        user_msg = f"CONTEXT:\n{context}\n\nQUESTION: {q['question']}"
        answer = chat_generate(model, tok, GEN_SYSTEM_PROMPT, user_msg)

        gold_retrieved = q['gold_section_id'] in retrieved_ids
        results[q['id']] = {
            'id': q['id'],
            'question': q['question'],
            'gold_answer': q['gold_answer'],
            'gold_section_id': q['gold_section_id'],
            'retrieved_ids': retrieved_ids,
            'gold_retrieved': gold_retrieved,
            'context': context,
            'generated_answer': answer,
        }
        save_json(results, save_path)
        print(f"[{phase_label}] {q['id']} done (gold_retrieved={gold_retrieved})")
    unload_current()
    return results

BASE_TOP_K = 3
baseline_generations = run_generation_phase(eval_set, dense_retrieve, BASE_TOP_K, BASELINE_GEN_PATH, 'baseline')
print(f'\nBaseline generation complete: {len(baseline_generations)}/{len(eval_set)} answered.')

## 6. Judging loop (baseline)

For each item: if `gold_retrieved` is False, label `retrieval-miss` programmatically (no LLM
call needed). Otherwise, ask the judge model to classify into one of
`correct / retrieved-but-ignored / retrieved-but-misused / hallucinated` with a strict rubric and
required JSON output including its reasoning (kept for spot-checking).


In [ ]:
JUDGE_SYSTEM_PROMPT = '''You are a strict, careful evaluator of RAG (retrieval-augmented generation) answers.
You will be given a QUESTION, the GOLD ANSWER (ground truth, verified against official documentation),
the CONTEXT that was retrieved and shown to the generating model, and the GENERATED ANSWER produced by that model.
The gold answer's supporting section IS present in the CONTEXT for this case (retrieval succeeded) -- you are
only deciding what the generating model did with it.

Classify the GENERATED ANSWER into exactly one of these four categories:
- "correct": the generated answer matches the gold answer's substance (paraphrase is fine), and is well
  supported by the context.
- "retrieved-but-ignored": the correct information is clearly present in the context, but the generated
  answer fails to use it -- e.g. it says "I don't know" or answers a different, unrelated question, or
  omits the specific fact that was asked for despite it being stated plainly in context.
- "retrieved-but-misused": the correct information is present in the context, but the generated answer
  misreads, misquotes, or misapplies it -- e.g. swaps a default value, inverts a condition, cites the
  wrong function from the same context, or draws an incorrect inference from correct source material.
- "hallucinated": the generated answer states specific facts (numbers, exceptions, defaults, behavior)
  that are not supported by the context and contradict the gold answer, with no indication the model
  noticed the context didn't support its claim.

Respond with ONLY a JSON object, no other text, in this exact form:
{"category": "<one of the four above>", "reasoning": "<1-3 sentences explaining your decision, quoting the specific detail that drove it>"}
'''

def parse_judge_json(raw):
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {'category': 'PARSE_ERROR', 'reasoning': raw[:300]}
    try:
        obj = json.loads(match.group(0))
        obj.setdefault('category', 'PARSE_ERROR')
        obj.setdefault('reasoning', '')
        return obj
    except Exception:
        return {'category': 'PARSE_ERROR', 'reasoning': raw[:300]}

def run_judging_phase(generations, save_path, phase_label):
    judgments = load_json(save_path, {})
    needs_llm = [g for g in generations.values() if g['id'] not in judgments and g['gold_retrieved']]
    if needs_llm:
        model, tok = load_judge()
    for g in generations.values():
        if g['id'] in judgments:
            continue
        if not g['gold_retrieved']:
            judgments[g['id']] = {'id': g['id'], 'category': 'retrieval-miss', 'reasoning': 'gold_section_id not in retrieved_ids (programmatic)'}
            save_json(judgments, save_path)
            continue
        user_msg = (
            f"QUESTION: {g['question']}\n\n"
            f"GOLD ANSWER: {g['gold_answer']}\n\n"
            f"CONTEXT:\n{g['context']}\n\n"
            f"GENERATED ANSWER: {g['generated_answer']}"
        )
        raw = chat_generate(model, tok, JUDGE_SYSTEM_PROMPT, user_msg, max_new_tokens=200)
        parsed = parse_judge_json(raw)
        judgments[g['id']] = {'id': g['id'], 'category': parsed['category'], 'reasoning': parsed['reasoning']}
        save_json(judgments, save_path)
        print(f"[{phase_label}] {g['id']} -> {parsed['category']}")
    if needs_llm:
        unload_current()
    return judgments

baseline_judgments = run_judging_phase(baseline_generations, BASELINE_JUDGE_PATH, 'baseline')
print(f'\nBaseline judging complete: {len(baseline_judgments)}/{len(eval_set)} judged.')

In [ ]:
from collections import Counter
print('Baseline category counts:')
print(Counter(j['category'] for j in baseline_judgments.values()))

## 7. Intervention: cross-encoder reranking

Simplest of the three options to isolate cleanly: retrieve a wide dense candidate pool (top-10),
rerank with a cross-encoder, keep top-3 for the generator. No re-embedding, no index rebuild, no
second retrieval path to fuse — the FAISS index from Section 3 is untouched. This targets cases
where the gold chunk was retrieved somewhere in the top-10 but ranked outside the original top-3;
it cannot fix a true miss where the gold chunk isn't in the top-10 at all (called out in the
analysis below).


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cpu')

def reranked_retrieve(query, k=3, candidate_pool=10):
    candidates = dense_retrieve(query, k=candidate_pool)
    if not candidates:
        return []
    pairs = [(query, c['chunk']['text']) for c in candidates]
    scores = reranker.predict(pairs)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)
    candidates.sort(key=lambda c: c['rerank_score'], reverse=True)
    return candidates[:k]

# smoke test
for r in reranked_retrieve('what does itertools.chain do', k=3):
    print(round(r['rerank_score'], 3), r['id'])

In [ ]:
intervention_generations = run_generation_phase(eval_set, reranked_retrieve, 3, INTERV_GEN_PATH, 'intervention')
print(f'\nIntervention generation complete: {len(intervention_generations)}/{len(eval_set)} answered.')

In [ ]:
intervention_judgments = run_judging_phase(intervention_generations, INTERV_JUDGE_PATH, 'intervention')
print('Intervention category counts:')
print(Counter(j['category'] for j in intervention_judgments.values()))

## 8. Before / after comparison chart

In [ ]:
import matplotlib.pyplot as plt

CATEGORIES = ['correct', 'retrieval-miss', 'retrieved-but-ignored', 'retrieved-but-misused', 'hallucinated', 'PARSE_ERROR']

def counts_for(judgments):
    c = Counter(j['category'] for j in judgments.values())
    return [c.get(cat, 0) for cat in CATEGORIES]

base_counts = counts_for(baseline_judgments)
interv_counts = counts_for(intervention_judgments)

x = np.arange(len(CATEGORIES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, base_counts, width, label='Baseline (dense top-3)')
ax.bar(x + width/2, interv_counts, width, label='Intervention (rerank top-3 of 10)')
ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES, rotation=20, ha='right')
ax.set_ylabel('# of eval questions')
ax.set_title('RAG Failure Modes: Baseline vs. Cross-Encoder Reranking')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/before_after_chart.png', dpi=150)
plt.show()

print('\nBaseline:    ', dict(zip(CATEGORIES, base_counts)))
print('Intervention:', dict(zip(CATEGORIES, interv_counts)))

## 9. Manual spot-check sample for the local judge

Local-model-as-judge (even a 7B model) is noisier than a frontier API judge — it can misparse
subtle "misused vs. hallucinated" distinctions, and its own instruction-following on the JSON
format isn't perfect (watch for `PARSE_ERROR` entries above). Before trusting the category counts
in Section 8, spot-check a stratified sample by hand.

This cell samples ~15-20 judged items (stratified across categories where possible, prioritizing
the harder-to-judge categories) from BOTH phases, and writes a CSV to Drive with the question,
context, generated answer, judge's category + reasoning, and two blank columns for you to fill in
by hand: `human_agree` (Y/N) and `human_category` (only if you disagree).


In [ ]:
import pandas as pd

def build_spotcheck_sample(n_total=18):
    rows = []
    all_items = []
    for phase, gens, judges in [('baseline', baseline_generations, baseline_judgments),
                                 ('intervention', intervention_generations, intervention_judgments)]:
        for qid, j in judges.items():
            g = gens[qid]
            all_items.append((phase, g, j))

    by_cat = {}
    for item in all_items:
        by_cat.setdefault(item[2]['category'], []).append(item)

    random.seed(42)
    per_cat = max(1, n_total // max(1, len(by_cat)))
    sample = []
    for cat, items in by_cat.items():
        sample.extend(random.sample(items, min(per_cat, len(items))))
    # top up to n_total if short
    remaining = [i for i in all_items if i not in sample]
    random.shuffle(remaining)
    while len(sample) < n_total and remaining:
        sample.append(remaining.pop())

    for phase, g, j in sample:
        rows.append({
            'phase': phase,
            'id': g['id'],
            'question': g['question'],
            'gold_answer': g['gold_answer'],
            'gold_retrieved': g['gold_retrieved'],
            'generated_answer': g['generated_answer'],
            'judge_category': j['category'],
            'judge_reasoning': j['reasoning'],
            'human_agree_Y_N': '',
            'human_category_if_disagree': '',
        })
    return pd.DataFrame(rows)

spot_df = build_spotcheck_sample(18)
spot_df.to_csv(SPOTCHECK_PATH, index=False)
print(f'Wrote {len(spot_df)} rows to {SPOTCHECK_PATH}')
spot_df[['phase', 'id', 'judge_category']]

**How to use this file:** open `spotcheck_sample.csv` from Drive (or download it), read each
question/context/answer/judge-reasoning row, and fill in `human_agree_Y_N`. Where you disagree,
note what you think the correct category is. Then run the cell below to compute the agreement
rate — this is the number to report in your written analysis as the reliability check on the
local judge, per your original ask.


In [ ]:
# Run this AFTER you've manually filled in the CSV (re-download/re-upload if editing locally).
spot_df_reviewed = pd.read_csv(SPOTCHECK_PATH)
reviewed = spot_df_reviewed[spot_df_reviewed['human_agree_Y_N'].astype(str).str.upper().isin(['Y', 'N'])]
if len(reviewed) == 0:
    print('No rows filled in yet — fill human_agree_Y_N with Y/N for at least a few rows, then re-run.')
else:
    agree_rate = (reviewed['human_agree_Y_N'].astype(str).str.upper() == 'Y').mean()
    print(f'Human/judge agreement on {len(reviewed)} spot-checked items: {agree_rate:.1%}')
    print(reviewed[reviewed['human_agree_Y_N'].astype(str).str.upper() == 'N'][['id', 'judge_category', 'human_category_if_disagree']])

## 10. Written analysis

*(Filled in from an actual completed run: 63 eval questions, both phases fully generated and judged, spot-check sample manually reviewed.)*

**1. Overall shift.** Contrary to what a reranker is "supposed" to do, total `correct` answers
*dropped slightly* from baseline to intervention: 55/63 (87.3%) → 52/63 (82.5%), a −3-point change.
Reranking did not net-improve answer quality on this corpus/eval-set combination.

**2. Retrieval-miss.** Retrieval-miss fell as expected: 2/63 → 1/63. Of the two baseline misses:
`q010` (a `re.match` case-insensitivity question) had its gold chunk sitting outside the baseline
top-3 but recoverable — reranking correctly pulled `re#re.match` into the top-3 from the wider
candidate pool, confirming the reranker works as designed *when the gold chunk is retrievable at
all*. `q036` (a `re.escape` question) stayed a miss in both phases — its gold chunk wasn't
surfaced even after reranking, meaning it likely wasn't in the dense top-10 candidate pool to
begin with. This is exactly the failure mode reranking cannot fix: it can only reorder candidates
already retrieved, not find candidates dense search missed entirely.

**3. Ignored vs. misused — the real story.** The interesting movement isn't in the totals, it's in
the transition matrix. Five questions changed category between phases:
- `q010`: retrieval-miss → retrieved-but-ignored — retrieval was fixed, but the generator then
  failed to *use* the newly-available correct chunk. Net effect on this question: no improvement,
  just a different failure mode.
- `q004`, `q059`: correct → retrieved-but-ignored — gold chunks were retrieved fine at baseline;
  reranking changed *which other* chunks accompanied it in the top-3, and the altered distractor
  context was apparently enough to make the 3B generator ignore an answer it had previously found.
- `q057`: correct → retrieved-but-misused — same pattern, different failure type.
- `q058`: retrieved-but-misused → hallucinated — this one got strictly worse in kind, not just in
  outcome.

Four of five swaps were regressions; only `q010` was a genuine retrieval fix. This points to a
specific mechanism worth stating explicitly: **reranking changed the composition of the top-3
context window, and the 3B generator turned out to be sensitive to that composition in ways that
sometimes hurt more than the improved ranking helped.** This is a useful finding for a failure-mode
study — it suggests the generator, not the retriever, is the bottleneck for a meaningful share of
these cases, consistent with the original design intent of using a deliberately weak generator to
expose exactly this kind of failure.

**4. Judge reliability.** Manual spot-check of 18 stratified items: **15/18 agreement (83.3%)**
between the human reviewer and the local Qwen2.5-7B judge. All three disagreements were boundary
calls between adjacent categories rather than wild misses:
- `q001`: judge said `retrieved-but-ignored`, human called it `retrieved-but-misused` (the answer
  engaged with the wrong part of the retrieved chunk rather than ignoring it outright).
- `q006`: judge said `retrieved-but-misused`, human called it `correct` (judge appears to have
  over-penalized a valid paraphrase).
- `q058` (intervention): judge said `hallucinated`, human called it `retrieved-but-misused` (the
  claim traced back to a misreading of real context rather than being fabricated from nothing).

83.3% falls in the "directionally correct but flag the confusion pattern" band from this notebook's
rule of thumb, not the "fully reliable" band. The `misused`-vs-`hallucinated` boundary is confirmed
here as the specific soft spot (it shows up in both `q001` and `q058`), so the −3-point overall
shift and the single `hallucinated` count in the intervention phase should be read with a grain of
salt rather than as an exact tally — it's plausible the true intervention `hallucinated` count is 0
and that item is actually `retrieved-but-misused`, per the human review.

**5. Limitations.** With 63 questions and single-digit category counts, each of the five category
swaps moves a percentage point by roughly 1.6 points — the entire "intervention got worse"
headline rests on a net swing of 3 questions out of 63, well within the range that a stricter or
more lenient judge, or a different random seed in generation, could plausibly reverse. This should
be read as a methodology demonstration of how to structure and instrument a RAG failure-mode eval,
not as a statistically powered claim that cross-encoder reranking hurts small-model RAG pipelines
in general. A useful follow-up would be increasing the eval set size and/or running each phase with
multiple generation seeds to see whether the `correct`-count regression replicates.
